In [1]:
!pip install --upgrade datasets huggingface_hub

from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

In [2]:
import pandas as pd
import numpy as np
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [3]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.shape)
print(train_df.head(2))
print(train_df["label"].value_counts())

(25000, 2)
                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0
label
0    12500
1    12500
Name: count, dtype: int64


In [4]:
train_df.isnull().sum()

,0
text,0
label,0


In [5]:
train_df["text_len"] = train_df["text"].str.len()

In [6]:
print(train_df["text_len"].describe())
print(train_df["text"].iloc[0])

count    25000.00000
mean      1325.06964
std       1003.13367
min         52.00000
25%        702.00000
50%        979.00000
75%       1614.00000
max      13704.00000
Name: text_len, dtype: float64
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher,

In [7]:
from bs4 import BeautifulSoup

In [8]:
train_df["text"] = train_df["text"].apply(
    lambda x: BeautifulSoup(x,"html.parser").get_text()
)

In [9]:
test_df["text"] = test_df["text"].apply(
    lambda x: BeautifulSoup(x,"html.parser").get_text()
)

In [10]:
print(train_df["text"].iloc[0])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot lik

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [12]:
tfidf = TfidfVectorizer(max_features=10000)
x_train = tfidf.fit_transform(train_df["text"])
x_test = tfidf.transform(test_df["text"])
y_train = train_df[["label"]].values.ravel()
y_test = test_df[["label"]].values.ravel()

In [13]:
y_train

array([0, 0, 0, ..., 1, 1, 1])

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *

In [15]:
LR = LogisticRegression()

model_lr = LR.fit(x_train,y_train)

y_pred = model_lr.predict(x_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.88      0.89      0.88     12500
           1       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [16]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")
joblib.dump(model_lr, "models/lr_model.pkl")

['models/lr_model.pkl']

In [17]:
import joblib

tfidf = joblib.load("/content/models/lr_model.pkl")
model_lr = joblib.load("/content/models/tfidf_vectorizer.pkl")

In [18]:
import transformers

In [19]:
from transformers import BertTokenizer, BertForSequenceClassification

In [20]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
encoding = tokenizer(
    train_df["text"].tolist(),
    truncation = True,
    padding = True,
    max_length = 128,
    return_tensors = "pt"
)

In [22]:
print(encoding.keys())

KeysView({'input_ids': tensor([[  101,  1045, 12524,  ...,  2679,  3314,   102],
        [  101,  1000,  1045,  ...,  8040,  7317,   102],
        [  101,  2065,  2069,  ...,     0,     0,     0],
        ...,
        [  101,  2023,  2143,  ...,  2362,  1012,   102],
        [  101,  1005,  1996,  ...,  7432,  2121,   102],
        [  101,  1996,  2466,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]])})


In [23]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMDbDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = IMDbDataset(encoding, train_df["label"].tolist())

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


encoding = tokenizer(
    test_df["text"].tolist(),
    truncation = True,
    padding = True,
    max_length = 128,
    return_tensors = "pt"
)

test_dataset = IMDbDataset(encoding, test_df["label"].tolist())

test_loader = DataLoader(test_dataset, batch_size=16)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

cuda


In [25]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),lr=2e-5)

epochs = 3

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    optimizer.zero_grad()

    outputs = model(input_ids = input_ids, attention_mask = attention_mask, labels = labels)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  print(f"Epochs {epoch+1} | Loss : {total_loss/len(train_loader):.4f}")

Epochs 1 | Loss : 0.3263
Epochs 2 | Loss : 0.1886
Epochs 3 | Loss : 0.0945


In [29]:
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
  for batch in test_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    outputs = model(input_ids = input_ids, attention_mask = attention_mask)
    preds = torch.argmax(outputs.logits, dim=1)

    all_preds.append(preds.cpu().numpy())
    all_labels.append(labels.cpu().numpy())


all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)

from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.93      0.79      0.85     12500
           1       0.82      0.95      0.88     12500

    accuracy                           0.87     25000
   macro avg       0.87      0.87      0.86     25000
weighted avg       0.87      0.87      0.86     25000



* Here we can see that BERT caused underperform to the base line.
* It that mean TF-IDF is better than BERT?.
* No, BERT not performing better because of we use only **3 epoch** with **lr = 2e-5**,if we use **warmup scheduled**(start small → increase to target lr → decrease gradually) and use **3 to 5 epochs**, we get upto **92-93%**.

In [30]:
model.save_pretrained("models/bert_model")
tokenizer.save_pretrained("models/bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('models/bert_model/tokenizer_config.json', 'models/bert_model/tokenizer.json')

In [31]:
#because i use colab
from google.colab import files
import shutil

shutil.make_archive("models", "zip", "models")
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>